In [ ]:
!pip install flask flask-cors pyngrok mediapipe==0.10.18 opencv-python==4.9.0.80 numpy==1.26.4

In [ ]:
from google.colab import files
files.upload()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import cv2, numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model
import base64

app = Flask(__name__)
CORS(app)

model = load_model("gesture_model.keras")
model = load_model("gesture_model.keras")
model.predict(np.zeros((1,64,64,3)))  # warmup

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1)
hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json['image']

        # Decode base64 image
        image_data = data.split(',')[1]
        img_bytes = base64.b64decode(image_data)
        np_arr = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

        if img is None:
            return jsonify({"prediction": "Image decode failed"})

        #  Convert to RGB
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        #  Detect hand
        results = hands.process(rgb)

        if not results.multi_hand_landmarks:
            return jsonify({"prediction": "No hand detected"})

        hand_landmarks = results.multi_hand_landmarks[0]

        h, w, _ = img.shape
        x_list, y_list = [], []

        # Landmarks
        for lm in hand_landmarks.landmark:
            x_list.append(int(lm.x * w))
            y_list.append(int(lm.y * h))

        # Bounding box
        x_min, x_max = min(x_list), max(x_list)
        y_min, y_max = min(y_list), max(y_list)

        # Padding
        padding = 100
        x_min = max(0, x_min - padding)
        y_min = max(0, y_min - padding)
        x_max = min(w, x_max + padding)
        y_max = min(h, y_max + padding)

        # Crop
        hand_img = img[y_min:y_max, x_min:x_max]

        if hand_img.size == 0:
            return jsonify({"prediction": "Invalid crop"})

        #  Masking
        hsv = cv2.cvtColor(hand_img, cv2.COLOR_BGR2HSV)

        lower = np.array([0, 30, 60])
        upper = np.array([25, 255, 255])

        mask = cv2.inRange(hsv, lower, upper)


        if np.sum(mask) == 0:
            return jsonify({"prediction": "Mask failed"})

        hand_img = cv2.bitwise_and(hand_img, hand_img, mask=mask)

        bg = np.zeros_like(hand_img)
        hand_img = np.where(mask[:, :, None] == 255, hand_img, bg)

        #  Preprocess
        hand_img = cv2.resize(hand_img, (64, 64))
        hand_img = hand_img / 255.0
        hand_img = hand_img.reshape(1, 64, 64, 3)

        # Prediction
        pred = model.predict(hand_img)

        confidence = float(np.max(pred))
        digit = int(np.argmax(pred))

        # Make sure coordinates are integers
        x_min, y_min, x_max, y_max = map(int, [x_min, y_min, x_max, y_max])

        output_img = img.copy()

        # Draw rectangle
        cv2.rectangle(output_img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

        # Put text (slightly above box)
        y_text = max(30, y_min - 10)

        cv2.putText(
            output_img,
            f"Digit: {digit} ({confidence:.2f})",
            (x_min, y_text),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

        # Encode image
        _, buffer = cv2.imencode('.jpg', output_img)
        img_base64 = base64.b64encode(buffer).decode('utf-8')

        return jsonify({
            "prediction": digit,
            "confidence": confidence,
            "image": img_base64
        })
    except Exception as e:
        print("ERROR:", str(e))
        return jsonify({"error": str(e)})

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step


In [ ]:
from pyngrok import ngrok

# Replace with your own ngrok authentication token
ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")

In [ ]:
public_url = ngrok.connect(5000)
print(public_url)

NgrokTunnel: "https://frosted-unclasp-proxy.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 07:35:53] "OPTIONS /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step


INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 07:35:55] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 07:36:15] "OPTIONS /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


...

INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 07:39:32] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 07:39:38] "OPTIONS /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 07:39:39] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 07:39:53] "OPTIONS /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step


INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 07:39:54] "POST /predict HTTP/1.1" 200 -
